# Recalibration

> How much of fine-tuning a linear correction of the zero-shot predictions, or a refit of the BatchNorm statistics or the head, can replace

In [ ]:
#| default_exp recalibrate

In [ ]:
#| export
import numpy as np, pandas as pd
from soilspecdl.all import BatchSNV, adabn, get_spectral_dls, predict, ridge_head
from ossl_rt_transfer.pretrain import rt_wns

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
from fastcore.test import *
from soilspecdl.all import SpectralCNN, regression_scores
from soilspecdata.datasets.ringtrial import load_ringtrial
from ossl_rt_transfer.data import rt_path, near_dup_groups
from ossl_rt_transfer.pretrain import load_pretrained, log_props
from ossl_rt_transfer.finetune import rt_xy, soil_folds, paired_test, cached

The cells that read the ring-trial data are marked `#| rtdata`. The outputs on this page come from a local run.

In `07_finetune`, fine-tuning on the soils of one instrument raised R² well above zero-shot, for K most of all. Fine-tuning can change every weight, so it can correct anything from an offset in the predictions to the features themselves. This notebook asks how much of its gain cheaper corrections give, each fitted on the same training soils and folds:

- `bias`: the zero-shot predictions plus the mean training residual.
- `slope and bias`: a linear regression of the targets on the zero-shot predictions.
- `AdaBN`: the BatchNorm statistics recomputed on the instrument's training spectra, with all weights fixed. It needs no targets.
- `AdaBN, slope and bias`: AdaBN followed by a linear regression of the targets on its predictions.
- `head refit`: the last linear layer refitted by ridge regression on the pooled features, with the backbone fixed.

If `slope and bias` comes close to fine-tuning, the pretrained model already ranks the soils of that instrument as well as fine-tuning does, and fine-tuning mostly recalibrates. A linear correction cannot change the correlation, so the squared zero-shot correlation is the most it can reach in R² on the training soils.

## Corrections

`linear_recal` maps predictions onto targets with a bias, or with a slope and bias, fitted on the training soils:

In [ ]:
#| export
def linear_recal(
    p_tr, # Predictions for the training soils
    y_tr, # Targets of the training soils
    p_te, # Predictions to correct
    slope=True, # Fit a slope as well as a bias
)->np.ndarray: # Corrected predictions
    "Correct `p_te` with the bias, or the slope and bias, that map `p_tr` onto `y_tr`"
    if not slope: return p_te + (y_tr - p_tr).mean()
    b, a = np.polyfit(p_tr, y_tr, 1)
    return a + b * p_te

In [ ]:
p = np.linspace(0, 1, 10)
test_close(linear_recal(p, 2 * p + 1, np.array([3.])), [7.])
test_close(linear_recal(p, p + 1, np.array([3.]), slope=False), [4.])

`adabn_model` recomputes the BatchNorm statistics from the training spectra only, so the held-out soils play no part:

In [ ]:
#| export
def adabn_model(
    model, # Pretrained model
    X, # SNV spectra on `rt_wns`, one per row
    train, # Indices of the training soils
):
    "Copy of `model` whose BatchNorm statistics are recomputed on the spectra `X[train]`"
    idx = np.arange(len(train))
    dls, _ = get_spectral_dls(X[train], np.zeros(len(train)), rt_wns, splitter=lambda X, y: (idx, idx, idx), batch_tfms=[BatchSNV()])
    return adabn(model, dls).cpu()

`cv_recal` gives the held-out predictions of each correction for every fold of every repeat, in the format of `cv_cnn`:

In [ ]:
#| export
def cv_recal(
    model, # Pretrained model
    X, # SNV spectra on `rt_wns`, one per row
    y, # Targets
    ids, # Sample id of each row
    folds, # Fold of each soil, from `soil_folds`
)->pd.DataFrame: # Held-out predictions
    "Held-out predictions of the bias, slope and bias, AdaBN and head-refit corrections, for every fold of every repeat"
    p0, preds = predict(model, X, rt_wns).ravel(), []
    for r, f in folds[ids].iterrows():
        f = f.values
        for k in np.unique(f):
            tr, te = np.flatnonzero(f != k), np.flatnonzero(f == k)
            pa = predict(adabn_model(model, X, tr), X, rt_wns).ravel()
            res = {'bias': linear_recal(p0[tr], y[tr], p0[te], slope=False),
                   'slope and bias': linear_recal(p0[tr], y[tr], p0[te]),
                   'AdaBN': pa[te],
                   'AdaBN, slope and bias': linear_recal(pa[tr], y[tr], pa[te]),
                   'head refit': predict(ridge_head(model, X[tr], y[tr], rt_wns), X[te], rt_wns).ravel()}
            for m, p in res.items():
                preds.append(pd.DataFrame({'method': m, 'repeat': r, 'fold': k, 'sample_id': ids[te], 'y': y[te], 'pred': p}))
    return pd.concat(preds, ignore_index=True)

On random spectra and targets, each correction predicts every soil once per repeat:

In [ ]:
rng = np.random.default_rng(0)
from ossl_rt_transfer.pretrain import preprocess
Xs = preprocess().fit_transform(rng.uniform(0.2, 2, (40, len(rt_wns))))
ys, gs = rng.normal(size=40), np.arange(40) // 2
ids = np.array([f'RT_{i}' for i in range(40)])
pc = cv_recal(SpectralCNN(), Xs, ys, ids, soil_folds(pd.Series(gs, index=ids), k=4, repeats=2))
test_eq(pc.groupby(['method', 'repeat']).sample_id.nunique().unique(), [40])
test_eq(pc.method.nunique(), 5)

## Data

The instruments, properties, starting models and folds are those of `07_finetune`. Its predictions for zero-shot, block-4 ridge, fine-tuning and PLSR are loaded from its saved results. PLSR does not depend on the starting model, so its predictions are repeated for each:

In [ ]:
#| rtdata
rt = load_ringtrial(rt_path())
groups = pd.Series(near_dup_groups(rt['KSSL']), index=rt['KSSL'].sample_ids)
props, insts = ['potassium_cmolkg', 'clay_perc'], ['KSSL', 'IAEA_aug2022']
starts = {'augmented 160': 'aug_ep160', 'augmented 20': 'aug_ep20', 'not augmented 80': 'snv_ep80'}
folds = soil_folds(groups)
preds = []
for i in insts:
    for p in props:
        X, y, ids = rt_xy(rt[i], p)
        tag = dict(instrument=i, property=p)
        plsr = cached(f'plsr_{i}_{p}', None)
        for s, f in starts.items():
            preds.append(plsr.assign(**tag, start=s))
            m = load_pretrained(f'ossl_{p}_{f}')
            preds.append(cached(f'cnn_{i}_{p}_{f}', None).assign(**tag, start=s))
            preds.append(cached(f'recal_{i}_{p}_{f}', lambda: cv_recal(m, X, y, ids, folds)).assign(**tag, start=s))
preds = pd.concat(preds, ignore_index=True)
methods = ['zero-shot', 'bias', 'AdaBN', 'slope and bias', 'AdaBN, slope and bias', 'head refit', 'ridge block 4', 'fine-tuned', 'PLSR']
test_eq(set(preds.method), set(methods))

## Scores against wet chemistry

Mean R² and CCC over the 10 repeats, each repeat pooling the held-out predictions of its 10 folds. For K, they are on the `log1p` scale.:

In [ ]:
#| rtdata
keys = ['instrument', 'property', 'start', 'method']
def repeat_scores(d):
    inv = np.expm1 if d.name[1] in log_props else None
    return pd.DataFrame({r: regression_scores(e.y, e.pred, inv=inv) for r, e in d.groupby('repeat')}).T
scores = preds.groupby(keys).apply(repeat_scores).groupby(keys).mean()
def by_method(col): return scores[col].unstack('method')[methods]
by_method('R²').round(3)

method                                          zero-shot  ...   PLSR
instrument   property         start                        ...       
IAEA_aug2022 clay_perc        augmented 160         0.271  ...  0.448
                              augmented 20          0.395  ...  0.448
                              not augmented 80     -2.729  ...  0.448
             potassium_cmolkg augmented 160        -2.122  ...  0.404
                              augmented 20         -3.282  ...  0.404
                              not augmented 80     -2.572  ...  0.404
KSSL         clay_perc        augmented 160         0.681  ...  0.807
                              augmented 20          0.685  ...  0.807
                              not augmented 80      0.758  ...  0.807
             potassium_cmolkg augmented 160         0.454  ...  0.145
                              augmented 20          0.285  ...  0.145
                              not augmented 80      0.205  ...  0.145

[12 rows x 9 columns]

In [ ]:
#| rtdata
by_method('CCC').round(3)

method                                          zero-shot  ...   PLSR
instrument   property         start                        ...       
IAEA_aug2022 clay_perc        augmented 160         0.637  ...  0.755
                              augmented 20          0.625  ...  0.755
                              not augmented 80      0.244  ...  0.755
             potassium_cmolkg augmented 160         0.358  ...  0.631
                              augmented 20          0.287  ...  0.631
                              not augmented 80      0.372  ...  0.631
KSSL         clay_perc        augmented 160         0.857  ...  0.906
                              augmented 20          0.846  ...  0.906
                              not augmented 80      0.891  ...  0.906
             potassium_cmolkg augmented 160         0.778  ...  0.537
                              augmented 20          0.725  ...  0.537
                              not augmented 80      0.739  ...  0.537

[12 rows x 9 columns]

## Share of the fine-tuning gain

For each repeat, the mean squared error on the model scale gives the share of the fine-tuning gain that a method recovers: (zero-shot error − method error) / (zero-shot error − fine-tuned error). A share of 1 matches fine-tuning, and above 1 beats it:

In [ ]:
#| rtdata
mse = preds.assign(se=(preds.pred - preds.y)**2).groupby(keys + ['repeat']).se.mean().unstack('method')
share = mse.drop(columns='PLSR').apply(lambda c: (mse['zero-shot'] - c) / (mse['zero-shot'] - mse['fine-tuned']))
share.groupby(['instrument', 'property', 'start']).mean()[[m for m in methods if m not in ('zero-shot', 'fine-tuned', 'PLSR')]].round(2)

method                                          bias  ...  ridge block 4
instrument   property         start                   ...               
IAEA_aug2022 clay_perc        augmented 160     0.42  ...           0.86
                              augmented 20     -0.06  ...           0.62
                              not augmented 80  0.76  ...           1.00
             potassium_cmolkg augmented 160     0.73  ...           1.03
                              augmented 20      0.83  ...           0.99
                              not augmented 80  0.74  ...           0.98
KSSL         clay_perc        augmented 160     0.61  ...           0.53
                              augmented 20      0.52  ...           0.46
                              not augmented 80  0.53  ...           0.03
             potassium_cmolkg augmented 160    -0.12  ...          -1.70
                              augmented 20      0.01  ...          -0.04
                              not augmented 80  0.02  ...           0.20

[12 rows x 6 columns]

## Paired tests against fine-tuning

As in `07_finetune`, the squared error on the model scale is averaged over the repeats and over the soils of each near-duplicate group, and each method is compared with fine-tuning from the same starting model. `A_better` is the share of groups where the method has the smaller error. The p-values are not corrected for the number of tests:

In [ ]:
#| rtdata
ge = (preds.assign(group=preds.sample_id.map(groups), se=(preds.pred - preds.y)**2)
      .groupby(keys + ['group']).se.mean())
vs_ft = pd.DataFrame({(i, p, s, m): paired_test(ge.loc[(i, p, s, m)], ge.loc[(i, p, s, 'fine-tuned')])
                      for i in insts for p in props for s in starts for m in ['slope and bias', 'head refit']}).T
vs_ft.index.names = keys
vs_ft[['A', 'B', 'A_better', 'p']].unstack('method').round(3)

A  ...               
method                                         head refit  ... slope and bias
instrument   property         start                        ...               
IAEA_aug2022 clay_perc        augmented 160        43.757  ...          0.002
                              augmented 20         49.064  ...          0.000
                              not augmented 80     60.941  ...          0.000
             potassium_cmolkg augmented 160         0.032  ...          0.160
                              augmented 20          0.036  ...          0.029
                              not augmented 80      0.033  ...          0.311
KSSL         clay_perc        augmented 160        13.214  ...          0.005
                              augmented 20         15.019  ...          0.009
                              not augmented 80      8.437  ...          0.136
             potassium_cmolkg augmented 160         0.028  ...          0.565
                              augmented 20          0.026  ...          0.006
                              not augmented 80      0.025  ...          0.253

[12 rows x 8 columns]

## Zero-shot calibration

A linear regression of the targets on the zero-shot predictions, over all soils, shows how far the pretrained model is from calibrated on each instrument. A calibrated model has a slope of 1 and predictions with the same mean as the targets. `r²` is the most R² a linear correction can reach on these soils:

In [ ]:
#| rtdata
def calib(d):
    b, a = np.polyfit(d.pred, d.y, 1)
    return pd.Series({'slope': b, 'mean pred - mean y': d.pred.mean() - d.y.mean(), 'r²': np.corrcoef(d.pred, d.y)[0, 1]**2})
zs = preds[(preds.method == 'zero-shot') & (preds.repeat == 0)]
cal = zs.groupby(['instrument', 'property', 'start']).apply(calib)
cal.join(scores['R²'].unstack('method')['fine-tuned'].rename('fine-tuned R²')).round(3)

slope  ...  fine-tuned R²
instrument   property         start                    ...               
IAEA_aug2022 clay_perc        augmented 160     0.803  ...          0.728
                              augmented 20      0.825  ...          0.728
                              not augmented 80  0.468  ...          0.687
             potassium_cmolkg augmented 160     0.438  ...          0.385
                              augmented 20      0.458  ...          0.448
                              not augmented 80  0.445  ...          0.508
KSSL         clay_perc        augmented 160     0.882  ...          0.911
                              augmented 20      0.916  ...          0.870
                              not augmented 80  0.864  ...          0.896
             potassium_cmolkg augmented 160     0.657  ...          0.602
                              augmented 20      0.600  ...          0.628
                              not augmented 80  0.561  ...          0.626

[12 rows x 4 columns]

## Results

**K: fine-tuning is a linear recalibration.** A slope and bias fitted on the zero-shot predictions gives R² 0.54 to 0.61 on KSSL and 0.34 to 0.44 on IAEA, against 0.60 to 0.63 and 0.39 to 0.51 for fine-tuning. It recovers 75% to 115% of the fine-tuning gain on KSSL and 97% to 98% on IAEA. In the paired tests it is not significantly worse than fine-tuning, except from the augmented 20 model (p = 0.006 on KSSL and 0.03 on IAEA). The squared correlation of the zero-shot predictions, 0.57 to 0.63 on KSSL and 0.38 to 0.47 on IAEA, is about the R² that fine-tuning reaches. So with these soils, fine-tuning does not rank the soils better than the pretrained model already does. It corrects the scale and the offset:
- The zero-shot predictions are too spread out. The slope of the targets on the predictions is 0.56 to 0.66 on KSSL and 0.44 to 0.46 on IAEA, where a calibrated model has 1.
- On IAEA they are also too high, by 0.33 to 0.42 on the `log1p` scale. On KSSL the offset is only 0.01 to 0.05.

The lower ceiling on IAEA, with r² 0.38 to 0.47 against 0.57 to 0.63, is the part of the K problem that comes from the IAEA spectra. PLSR trained on IAEA reaches no more (0.40).

**Clay: fine-tuning re-weights the features.** A slope and bias recovers only 53% to 66% of the fine-tuning gain on KSSL and is significantly worse than fine-tuning in two of three cases (p = 0.005 and 0.009). On IAEA it reaches R² 0.21 to 0.47 against 0.69 to 0.73 (p ≤ 0.002). Refitting the head on the pooled features does much better: R² 0.90 to 0.94 on KSSL, matching or beating fine-tuning, and 0.58 to 0.70 on IAEA. From the unaugmented model on KSSL it beats fine-tuning (0.94 against 0.90, p = 0.047). On IAEA, fine-tuning still beats the head refit from two of the three starting models (p = 0.003 and 0.009). So the pooled features of the pretrained clay models carry what is needed on KSSL, and mostly on IAEA, but the OSSL head combines them in a way that does not suit these instruments. On IAEA, part of the gain needs changes inside the backbone.

**AdaBN does not help.** Recomputing the BatchNorm statistics on 63 soils removes part of the IAEA offset without targets, but a bias correction does better. AdaBN lowers zero-shot R² in 4 of the 12 cases, down to -0.51 for IAEA clay from the augmented 20 model. Once a slope and bias are fitted, AdaBN lowers R² in 10 of the 12 cases and raises it by at most 0.004.

The two properties therefore need different things. For K, the pretrained model sets the ranking, and the soils of an instrument only serve to calibrate it. A few soils with a linear correction may be enough, and better K needs better features, not more fine-tuning. For clay, the soils of an instrument re-weight the features, and on IAEA they also change the backbone.